<a href="https://colab.research.google.com/github/DHARSHINIKANDASAMY/ResearchPaper_project/blob/main/P2(data_preprocessing).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
train_path = "/content/drive/MyDrive/FIRE2026/train_subtask1.json"
dev_path = "/content/drive/MyDrive/FIRE2026/dev_subtask1.json"

with open(train_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(dev_path, "r", encoding="utf-8") as f:
    dev_data = json.load(f)

train_df = pd.DataFrame(train_data)
dev_df = pd.DataFrame(dev_data)

print(train_df.head())

        ID                                               Text  \
0  S1/2072  ममता बनर्जी ने किसी भी मंत्री को पद से नहीं हट...   
1  S1/1719  প্রতি ঘণ্টায় চাই হোম কোয়ারেন্টাইনদের ভিডিও! নয়...   
2  S1/5113  করোনা সংক্রমণের আতঙ্ক, ভিনরাজ্যে কর্মরত যুবককে...   
3  S1/1671  जम्मू-कश्मीर में हालात सुधारने और शांति वापस ल...   
4  S1/5140  It's immaterial whether Rahul or Modi becomes ...   

                                            Evidence     Label  
0  पश्चिम बंगाल की मुख्यमंत्री ममता बनर्जी ने राज...   REFUTES  
1  কর্ণাটক সরকারের নয়া সিদ্ধান্ত। কোয়ারেন্টাইনে থ...   REFUTES  
2  জানা গিয়েছে, বালুরঘাট থানার বোল্লা আদিবাসী পাড়...  SUPPORTS  
3  कांग्रेस ने जम्मू-कश्मीर में बिगड़ते हालात और ...  SUPPORTS  
4  Delhi Chief Minister Arvind Kejriwal on Friday...  SUPPORTS  


In [ ]:
print("Training Dataset")
print(train_df.isnull().sum())

print("\nValidation Dataset")
print(dev_df.isnull().sum())

Training Dataset
ID          0
Text        0
Evidence    0
Label       0
dtype: int64

Validation Dataset
ID          0
Text        0
Evidence    0
Label       0
dtype: int64


In [ ]:
print("Before:", len(train_df))

train_df = train_df.drop_duplicates(
    subset=["Text", "Evidence"]
)

print("After :", len(train_df))

Before: 4536
After : 4536


In [ ]:
import unicodedata

def normalize_unicode(text):
    return unicodedata.normalize("NFKC", str(text))

In [ ]:
train_df["Text"] = train_df["Text"].apply(normalize_unicode)
train_df["Evidence"] = train_df["Evidence"].apply(normalize_unicode)

dev_df["Text"] = dev_df["Text"].apply(normalize_unicode)
dev_df["Evidence"] = dev_df["Evidence"].apply(normalize_unicode)

In [ ]:
def clean_spaces(text):

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [ ]:
train_df["Text"] = train_df["Text"].apply(clean_spaces)
train_df["Evidence"] = train_df["Evidence"].apply(clean_spaces)

dev_df["Text"] = dev_df["Text"].apply(clean_spaces)
dev_df["Evidence"] = dev_df["Evidence"].apply(clean_spaces)

In [ ]:
def remove_invisible(text):

    text = text.replace("\u200b", "")

    text = text.replace("\ufeff", "")

    return text

In [ ]:
train_df["Text"] = train_df["Text"].apply(remove_invisible)
train_df["Evidence"] = train_df["Evidence"].apply(remove_invisible)

dev_df["Text"] = dev_df["Text"].apply(remove_invisible)
dev_df["Evidence"] = dev_df["Evidence"].apply(remove_invisible)

In [ ]:
encoder = LabelEncoder()

train_df["label"] = encoder.fit_transform(train_df["Label"])

dev_df["label"] = encoder.transform(dev_df["Label"])

print(encoder.classes_)

['REFUTES' 'SUPPORTS']


In [ ]:
train_df = train_df[[
    "Text",
    "Evidence",
    "label"
]]

dev_df = dev_df[[
    "Text",
    "Evidence",
    "label"
]]

In [ ]:
train_df.columns = [
    "claim",
    "evidence",
    "label"
]

dev_df.columns = [
    "claim",
    "evidence",
    "label"
]

In [ ]:
train_df.head()

,claim,evidence,label
0,ममता बनर्जी ने किसी भी मंत्री को पद से नहीं हट...,पश्चिम बंगाल की मुख्यमंत्री ममता बनर्जी ने राज...,0
1,প্রতি ঘণ্টায় চাই হোম কোয়ারেন্টাইনদের ভিডিও! ...,কর্ণাটক সরকারের নয়া সিদ্ধান্ত। কোয়ারেন্টাইনে...,0
2,"করোনা সংক্রমণের আতঙ্ক, ভিনরাজ্যে কর্মরত যুবককে...","জানা গিয়েছে, বালুরঘাট থানার বোল্লা আদিবাসী পা...",1
3,जम्मू-कश्मीर में हालात सुधारने और शांति वापस ल...,कांग्रेस ने जम्मू-कश्मीर में बिगड़ते हालात और ...,1
4,It's immaterial whether Rahul or Modi becomes ...,Delhi Chief Minister Arvind Kejriwal on Friday...,1


In [ ]:
train_df["claim_words"] = train_df["claim"].apply(
    lambda x: len(x.split())
)

train_df["evidence_words"] = train_df["evidence"].apply(
    lambda x: len(x.split())
)

print(train_df[["claim_words", "evidence_words"]].describe())

       claim_words  evidence_words
count  4536.000000     4536.000000
mean     15.579145      112.485891
std       4.847068       44.933392
min       6.000000        3.000000
25%      12.000000       82.000000
50%      15.000000      103.000000
75%      18.250000      135.000000
max      41.000000      323.000000


In [ ]:
train_df.to_csv(
    "/content/drive/MyDrive/FIRE2026/train_processed.csv",
    index=False
)

dev_df.to_csv(
    "/content/drive/MyDrive/FIRE2026/dev_processed.csv",
    index=False
)

In [ ]:


df = pd.read_csv("/content/drive/MyDrive/FIRE2026/train_processed.csv")
print(df.columns)
print(df.head())

Index(['claim', 'evidence', 'label', 'claim_words', 'evidence_words'], dtype='object')
                                               claim  \
0  ममता बनर्जी ने किसी भी मंत्री को पद से नहीं हट...   
1  প্রতি ঘণ্টায় চাই হোম কোয়ারেন্টাইনদের ভিডিও! ...   
2  করোনা সংক্রমণের আতঙ্ক, ভিনরাজ্যে কর্মরত যুবককে...   
3  जम्मू-कश्मीर में हालात सुधारने और शांति वापस ल...   
4  It's immaterial whether Rahul or Modi becomes ...   

                                            evidence  label  claim_words  \
0  पश्चिम बंगाल की मुख्यमंत्री ममता बनर्जी ने राज...      0           23   
1  কর্ণাটক সরকারের নয়া সিদ্ধান্ত। কোয়ারেন্টাইনে...      0           10   
2  জানা গিয়েছে, বালুরঘাট থানার বোল্লা আদিবাসী পা...      1           10   
3  कांग्रेस ने जम्मू-कश्मीर में बिगड़ते हालात और ...      1           17   
4  Delhi Chief Minister Arvind Kejriwal on Friday...      1           14   

   evidence_words  
0             107  
1              54  
2              96  
3             120  
4             184  